# Sparse-scaling benchmark: per-cell observables

A cell doubled `d` times is still bulk Si, so every property per primitive cell must be the same
at every size. This notebook compares each finished case of one sparse pipeline with two
references:

- **the reference case** (`reference_d00`): the dense pipeline on the primitive cell, without
  `doubling_Hamiltonian`, on the full `base_nfft` mesh;
- **the previous size** of the same pipeline (`d-1`).

The deviation from the reference also contains what the sparse settings change on purpose: the
bond cutoff (`bond_order`) shifts the bands slightly, and an interior window leaves out the states
outside it. Those effects are the same at every `d`. The deviation from the previous size isolates
what doubling changes, which should be zero.

[observables.py](observables.py) reads the output files and normalizes them per primitive cell.
Its docstring lists every curve. Two of them need care:

- `inv_mass_dos` is the smeared density of inverse effective masses, built from
  `effective_masses_0.dat`. The per-state file itself cannot be compared line by line.
- `epsi_*` of an energy-window run is cut below the window top. The sparse sum runs over the
  window bands only, so ε' is not compared at all.

It needs no `collect.py` run, so it works while the ladder is still going.

In [ ]:
import glob
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, os.path.abspath('.'))
import observables
from config import CONFIG

ROOT = os.environ.get('BENCH_OUTPUT_ROOT', CONFIG['output_root'])
PIPELINE = 'sparse'  # any sparse pipeline of config.py
REFERENCE = 'reference_d00'
TOL = 1e-3  # relative L2 deviation counted as "unchanged"

# palette: deep blue, vermillion, amber, then Okabe-Ito
PALETTE = ['#004488', '#D55E00', '#FFC20A', '#009E73', '#56B4E9', '#CC79A7', '#000000']
plt.rcParams.update({'font.size': 10, 'axes.grid': True, 'grid.alpha': 0.3})


def finished(path):
    rec = observables.load_record(path)
    if rec.get('status') != 'ok':
        print('skipping %s: status %s' % (os.path.basename(path), rec.get('status')))
        return None
    return rec


cases = []
for path in sorted(glob.glob(os.path.join(ROOT, PIPELINE + '_d[0-9][0-9]'))):
    rec = finished(path)
    if rec is not None:
        cases.append({'d': rec['case']['d'], 'rec': rec, 'curves': observables.load(path)})
ref_path = os.path.join(ROOT, REFERENCE)
ref = observables.load(ref_path) if os.path.exists(ref_path) and finished(ref_path) else {}
if not ref:
    print('no finished %s yet: only the size-to-size comparison is shown' % REFERENCE)

COLOR = {c['d']: PALETTE[i % len(PALETTE)] for i, c in enumerate(cases)}
STYLE = {c['d']: ('-', '--', ':')[i // len(PALETTE)] for i, c in enumerate(cases)}
names = sorted({n for c in cases for n in c['curves']})

pd.DataFrame(
    {
        'd': c['d'],
        'nawf': c['rec']['hamiltonian']['nawf'],
        'nk': c['rec']['mesh']['nkpnts'],
        'N x nk': 2 ** c['d'] * c['rec']['mesh']['nkpnts'],
        'nev': c['rec'].get('solver', {}).get('nev'),
        'skipped': ', '.join(s[0] for s in c['rec'].get('solver', {}).get('skipped', [])),
    }
    for c in cases
).set_index('d')

## 1. Every curve, every size

The colored lines are the sizes of the pipeline and the dashed black line is the reference.
Transport and dielectric curves show the `xx` component. The `yy` and `zz` components are in the
deviation tables below. A grey vertical line marks where a curve is cut for the comparison.

In [ ]:
shown = [n for n in names if not n.endswith(('_yy', '_zz'))]
ncol = 3
nrow = -(-len(shown) // ncol)
fig, axes = plt.subplots(nrow, ncol, figsize=(15, 3.2 * nrow), squeeze=False)
for ax, name in zip(axes.flat, shown):
    for c in cases:
        if name in c['curves']:
            cv = c['curves'][name]
            kw = dict(color=COLOR[c['d']], ls=STYLE[c['d']], lw=1.1)
            ax.plot(cv['x'], cv['y'], label='d=%d' % c['d'], **kw)
            if cv['xmax'] is not None:
                ax.axvline(cv['xmax'], color='#888888', lw=0.8)
    if name in ref:
        ax.plot(ref[name]['x'], ref[name]['y'], 'k--', lw=1.0, label='reference')
    ax.set_title(name)
for ax in axes.flat[len(shown) :]:
    ax.axis('off')
axes.flat[0].legend(fontsize=7, ncol=2)
fig.tight_layout()

## 2. Deviation against size

This is the relative L2 deviation of each curve, on the range it shares with the other curve,
using `observables.deviation`. Solid lines are the deviation from the reference and dashed lines
the deviation from the previous size. One panel per family. The dotted line is `TOL`.

What to expect:

- **vs reference:** flat in `d`, at the level the bond cutoff and the window set.
- **vs previous size:** small, and it should not grow. A dashed line that stays above `TOL` means
  the doubled cell gives a different answer from the smaller one.

In [ ]:
rows = []
for prev, c in zip([None] + cases[:-1], cases):
    row = {'d': c['d'], 'nawf': c['rec']['hamiltonian']['nawf']}
    for name, cv in c['curves'].items():
        if name in ref:
            row[name + ' vs ref'] = observables.deviation(cv, ref[name])
        if prev is not None and prev['d'] == c['d'] - 1 and name in prev['curves']:
            row[name + ' vs prev'] = observables.deviation(cv, prev['curves'][name])
    rows.append(row)
dev = pd.DataFrame(rows).set_index('d')

FAMILIES = {
    'DoS': ('dos', 'pdos_'),
    'transport': ('sigma_', 'seebeck_', 'kappa_', 'pf_'),
    'conductivity': ('cond_',),
    'doping': ('doping_',),
    'dielectric': ('epsi_',),
    'band curvature': ('hall_', 'inv_mass'),
}
families = {f: [n for n in names if n.startswith(p)] for f, p in FAMILIES.items()}
families = {f: members for f, members in families.items() if members}
fig, axes = plt.subplots(1, len(families), figsize=(4.6 * len(families), 4.2), squeeze=False)
for ax, (family, members) in zip(axes[0], families.items()):
    for i, name in enumerate(members):
        color = PALETTE[i % len(PALETTE)]
        for suffix, ls, mfc in ((' vs ref', '-', None), (' vs prev', '--', 'none')):
            col = name + suffix
            if col in dev and dev[col].notna().any():
                label = name if suffix == ' vs ref' or name + ' vs ref' not in dev else None
                ax.plot(dev['nawf'], dev[col], 'o' + ls, color=color, mfc=mfc, ms=4, label=label)
    ax.axhline(TOL, ls=':', color='k', lw=1)
    ax.set_xscale('log', base=2)
    ax.set_yscale('log')
    ax.set_xlabel('nawf')
    ax.set_title(family)
    ax.legend(fontsize=7)
axes[0][0].set_ylabel('relative L2 deviation')
fig.tight_layout()

In [ ]:
# per curve: the deviation from the reference at the largest size, and the
# worst deviation from the previous size
vs_ref = {
    c[: -len(' vs ref')]: dev[c].dropna().iloc[-1]
    for c in dev
    if c.endswith(' vs ref') and dev[c].notna().any()
}
vs_prev = {c[: -len(' vs prev')]: dev[c].max() for c in dev if c.endswith(' vs prev')}
summary = pd.DataFrame({'vs ref, largest d': vs_ref, 'vs prev, worst': vs_prev})
summary['vs prev within TOL'] = summary['vs prev, worst'] <= TOL
summary

## 3. Cubic symmetry of σ

Bulk Si is cubic, so σ_yy/σ_xx = σ_zz/σ_xx = 1 per cell. The doublings go along one or two axes
at a time, so a doubled cell is not cubic, but its per-cell tensor still has to be. The ratios
are averaged over the μ range where σ_xx is above 1% of its maximum.

In [ ]:
rows = []
for c in cases:
    cv = c['curves']
    if 'sigma_xx' not in cv:
        continue
    s = {k: cv['sigma_' + k]['y'] for k in ('xx', 'yy', 'zz')}
    keep = s['xx'] > 0.01 * s['xx'].max()
    rows.append(
        {
            'd': c['d'],
            'doublings': tuple(c['rec']['case']['doublings']),
            'σ_yy/σ_xx': np.mean(s['yy'][keep] / s['xx'][keep]),
            'σ_zz/σ_xx': np.mean(s['zz'][keep] / s['xx'][keep]),
        }
    )
pd.DataFrame(rows).set_index('d')

## 4. Where the time goes

These are PAOFLOW's own module timings (the lines of `logs/*.out`), per case. The left panel is
the share of the wall time per module. The right panel is the mesh pass per k-point that one pool
handles, with an n³ guide (the LAPACK scaling). Because `nk` falls as 1/n while each k-point costs
n³, the total grows as n².

In [ ]:
mods = []
for c in cases:
    rec = c['rec']
    row = {'d': c['d'], 'nawf': rec['hamiltonian']['nawf'], 'nk': rec['mesh']['nkpnts']}
    row['npool'] = rec['mesh']['npool']
    for m in rec['modules']:
        row[m['name']] = row.get(m['name'], 0.0) + m['seconds']
    mods.append(row)
mods = pd.DataFrame(mods).fillna(0.0).set_index('d')
modules = [k for k in mods if k not in ('nawf', 'nk', 'npool')]
share = mods[modules].div(mods[modules].sum(axis=1), axis=0)
big = [k for k in modules if share[k].max() > 0.02]
share['other'] = share[[k for k in modules if k not in big]].sum(axis=1)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.6), gridspec_kw={'wspace': 0.55})
bottom = np.zeros(len(share))
colors = {k: PALETTE[i % len(PALETTE)] for i, k in enumerate(big)}
colors['other'] = '#BBBBBB'
for k in big + ['other']:
    axes[0].bar(share.index, share[k], bottom=bottom, color=colors[k], label=k)
    bottom += share[k].values
axes[0].set_xlabel('d')
axes[0].set_ylabel('share of wall time')
axes[0].legend(fontsize=8, loc='upper left', bbox_to_anchor=(1.0, 1.0))

ax = axes[1]
if 'Sparse mesh pass' in mods:
    per_k = mods['Sparse mesh pass'] * mods['npool'] / mods['nk']
    ax.plot(mods['nawf'], per_k, 'o-', color=PALETTE[0], label='mesh pass, s per k per pool')
    x = mods['nawf'].values
    ax.plot(x, per_k.iloc[-1] * (x / x[-1]) ** 3, ':', color='k', lw=1, label='n$^3$')
ax.set_xscale('log', base=2)
ax.set_yscale('log')
ax.set_xlabel('nawf')
ax.set_ylabel('seconds')
ax.legend(fontsize=8)
mods[['nawf', 'nk'] + big].round(1)